# Build Longitudinal Cohort

Dans ce notebook, nous construisons la cohorte longitudinale à partir des données brutes de la cohorte Genkyst. Les données sont nettoyées, les variables cliniques et génétiques sont intégrées, la cible de survie ainsi que les variables dépendantes du temps sont définies, puis les patients sont répartis entre une cohorte de développement et une cohorte de validation externe.

À l'issue de cette étape, deux jeux de données sont générés :

- **adpkd_longitudinal_development.parquet**
- **adpkd_longitudinal_external_test.parquet**

Ces deux jeux de données serviront de point de départ pour la préparation des jeux d'entraînement et de test, la construction des snapshots longitudinaux, puis le développement et l'évaluation des modèles de survie.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

from pathlib import Path

### Importation datasets

In [ ]:
# ==========================================================
# CHARGEMENT DE DONNÉES
# ==========================================================

inclusion = pl.read_parquet("Preprocessed_data/inclusion.parquet")
image = pl.read_parquet("Preprocessed_data/image.parquet")
creat = pl.read_parquet("Preprocessed_data/creat.parquet")
genetics = pl.read_parquet("Preprocessed_data/genetics.parquet")
center = pl.read_parquet("Preprocessed_data/center.parquet")
last_follow_up = pl.read_parquet("Preprocessed_data/last_follow_up.parquet")

### Préparation de la base de données


In [ ]:
# ----------------------------------------------------------
# Nettoyage de la cohorte
# ----------------------------------------------------------

# Suppression des patients dont le sexe n'est pas renseigné
inclusion_clean = inclusion.filter(
    pl.col("sex").is_not_null()
)


# ----------------------------------------------------------
# Construction de la base longitudinale
# ----------------------------------------------------------

# Association des mesures biologiques avec les informations
# cliniques de chaque patient
df_long = (
    creat.select([
        "patient_id",
        "creat_date",
        "age_creat",
        "ckd_epi",
        "ekfc"
    ])
    .join(
        inclusion_clean.select([
            "patient_id",
            "center_id",
            "birth_date",
            "inclusion_date",
            "sex",
            "has_hypertension",
            "age_hypertension",
            "has_hypertension_before_35",
            "has_urologic_event",
            "age_cysts_pain",
            "age_cysts_infection",
            "age_intra_cysts_hemorrhage",
            "age_lithiasis",
            "age_macro_hematuria",
            "has_urologic_event_before_35",
        ]),
        on="patient_id",
        how="left"
    )
    .sort([
        "patient_id",
        "creat_date"
    ])
)

print(
    "Patients supprimés :",
    inclusion.height - inclusion_clean.height
)

In [ ]:
# ==========================================================
# AJOUT DES DONNÉES GÉNÉTIQUES
# ==========================================================

# ----------------------------------------------------------
# Nettoyage et recodage des informations génétiques
# ----------------------------------------------------------

genetics_clean = (
    genetics

    # Les patients sans information génétique (gene et gene_subgroup absents)
    # sont exclus. Les patients avec une information génétique incomplète
    # (variant non renseigné) sont conservés et classés dans la catégorie "other".
    .filter(
        ~(
            pl.col("gene").is_null() &
            pl.col("gene_subgroup").is_null()
        )
    )

    .with_columns([

        # Type de variant (truncating / missense / inconnu)

        pl.when(
            pl.col("variant").is_null() |
            (pl.col("variant").str.strip_chars() == "")
        )
        .then(pl.lit("inconnu"))

        .when(
            pl.col("variant")
            .str.to_lowercase()
            .is_in(["tru"])
        )
        .then(pl.lit("truncating"))

        .when(
            pl.col("variant").str.contains(
                r"(?i)del|dup|ins|fs|\*|splice|\+\d+|\-\d+"
            )
        )
        .then(pl.lit("truncating"))

        .otherwise(pl.lit("missense"))
        .alias("variant_type"),

        # Catégorie génétique utilisée par les modèles

        # Absence de variant
        pl.when(
            pl.col("variant").is_null() |
            (pl.col("variant").str.strip_chars() == "")
        )
        .then(pl.lit("other"))

        # Cas génétiques non résolus
        .when(pl.col("status") != "solved")
        .then(pl.lit("GUR"))

        # PKD2
        .when(pl.col("gene_subgroup") == "PKD2")
        .then(pl.lit("PKD2"))

        # PKD1 truncating
        .when(
            (pl.col("gene_subgroup") == "PKD1") &
            (pl.col("is_truncating"))
        )
        .then(pl.lit("PKD1_truncating"))

        # PKD1 non truncating
        .when(pl.col("gene_subgroup") == "PKD1")
        .then(pl.lit("PKD1_non_truncating"))

        # Autres situations
        .otherwise(pl.lit("other"))
        .alias("genetic")

    ])
)

# ----------------------------------------------------------
# Harmonisation des identifiants patients
# ----------------------------------------------------------

df_long = df_long.with_columns(
    pl.col("patient_id")
      .str.strip_chars()
      .str.to_uppercase()
)

genetics_clean = genetics_clean.with_columns(
    pl.col("patient_id")
      .str.strip_chars()
      .str.to_uppercase()
)

# ----------------------------------------------------------
# Ajout des informations génétiques
# ----------------------------------------------------------

df_long_genetics = (
    df_long
    .join(
        genetics_clean.select([
            "patient_id",
            "variant",
            "variant_type",
            "genetic"
        ]),
        on="patient_id",
        how="left"
    )
)

# ----------------------------------------------------------
# Conservation des patients avec une information génétique
# ----------------------------------------------------------

df_long_genetics = (
    df_long_genetics
    .filter(
        pl.col("genetic").is_not_null()
    )
)

In [ ]:
# ==========================================================
# AJOUT DES DONNÉES DE SUIVI
# ==========================================================

# Ajout des informations de suivi nécessaires à l'analyse de survie
df_survival_long = (
    df_long_genetics
    .join(
        last_follow_up.select([
            "patient_id",
            "dialysis_date",
            "transplant_date",
            "kidney_failure_date",
            "last_follow_up_date",
            "is_alive",
            "death_date",
            "last_creat_date",
            "is_on_renal_replacement"
        ]),
        on="patient_id",
        how="left"
    )
)


# ==========================================================
# DÉFINITION DE LA DATE DE L'ÉVÉNEMENT IRT
# ==========================================================

# La date d'événement correspond à la première occurrence parmi :
# - insuffisance rénale terminale,
# - début de dialyse,
# - transplantation rénale.
df_survival_long = df_survival_long.with_columns(

    pl.min_horizontal([
        pl.col("kidney_failure_date"),
        pl.col("dialysis_date"),
        pl.col("transplant_date")
    ]).alias("event_date")

)


# ==========================================================
# CRÉATION DE L'INDICATEUR D'ÉVÉNEMENT
# ==========================================================

# event_irt = 1 si une IRT est observée, 0 sinon
df_survival_long = df_survival_long.with_columns(

    pl.col("event_date")
    .is_not_null()
    .cast(pl.Int8)
    .alias("event_irt")

)


# ==========================================================
# CONSERVATION DES MESURES AVANT L'ÉVÉNEMENT
# ==========================================================

# Les mesures réalisées le jour de l'IRT ou après sont exclues.
df_survival_long = df_survival_long.filter(

    (pl.col("event_date").is_null()) |
    (pl.col("creat_date") < pl.col("event_date"))

)

In [ ]:
# ==========================================================
# ÂGE DE LA PREMIÈRE COMPLICATION UROLOGIQUE
# ==========================================================

# Âge de survenue de la première complication urologique
df_survival_long = df_survival_long.with_columns(

    pl.min_horizontal([
        "age_cysts_pain",
        "age_cysts_infection",
        "age_intra_cysts_hemorrhage",
        "age_lithiasis",
        "age_macro_hematuria"
    ]).alias("age_urologic_event")

)


# ==========================================================
# DÉFINITION DU SUIVI ET DU TEMPS DE SURVIE
# ==========================================================

df_survival_long = (
    df_survival_long

    # --------------------------------------------------
    # Nettoyage des dates incohérentes après le décès
    # --------------------------------------------------

    .with_columns([

        # Date de dernier suivi
        pl.when(
            (pl.col("death_date").is_not_null()) &
            (pl.col("last_follow_up_date") > pl.col("death_date"))
        )
        .then(None)
        .otherwise(pl.col("last_follow_up_date"))
        .alias("last_follow_up_date_clean"),

        # Dernière mesure biologique
        pl.when(
            (pl.col("death_date").is_not_null()) &
            (pl.col("last_creat_date") > pl.col("death_date"))
        )
        .then(None)
        .otherwise(pl.col("last_creat_date"))
        .alias("last_creat_date_clean")

    ])

    # --------------------------------------------------
    # Définition de la date de fin de suivi
    # --------------------------------------------------

    .with_columns(

        pl.when(pl.col("event_irt") == 1)
        .then(pl.col("event_date"))

        .when(pl.col("death_date").is_not_null())
        .then(pl.col("death_date"))

        .otherwise(
            pl.max_horizontal([
                pl.col("last_follow_up_date_clean"),
                pl.col("last_creat_date_clean")
            ])
        )

        .alias("end_date")

    )

    # --------------------------------------------------
    # Calcul du temps jusqu'à l'événement ou la censure
    # --------------------------------------------------

    .with_columns(

        (
            (pl.col("end_date") - pl.col("creat_date"))
            .dt.total_days() / 365.25
        )
        .alias("time_to_event")

    )
)

print("Taille du jeu de données :", df_survival_long.shape)

print(
    "Nombre de patients :",
    df_survival_long.select(
        pl.col("patient_id").n_unique()
    )
)

In [ ]:
# ==========================================================
# SÉLECTION DE LA COHORTE D'ÉTUDE
# ==========================================================

# --------------------------------------------------
# Effectifs initiaux
# --------------------------------------------------

n_patients_initial = (
    df_survival_long
    .select(pl.col("patient_id").n_unique())
    .item()
)

n_obs_initial = df_survival_long.height


# -------------------------------------------------------------------
# Conservation des visites réalisées à partir de 16 ans
# -------------------------------------------------------------------

df_adult = (
    df_survival_long
    .filter(
        pl.col("age_creat") >= 16
    )
)


# --------------------------------------------------
# Conservation des visites avec données biologiques
# --------------------------------------------------

df_biology = (
    df_adult
    .filter(
        pl.col("creat_date").is_not_null()
        & pl.col("ckd_epi").is_not_null()
    )
)


# --------------------------------------------------
# Prédiction avant le stade CKD4 avancé
# --------------------------------------------------

# Les observations avec un DFG < 30 mL/min/1.73 m² sont
# exclues afin de prédire l'évolution avant l'insuffisance
# rénale avancée.

df_ckd = (
    df_biology
    .filter(
        pl.col("ckd_epi") >= 30
    )
)


# --------------------------------------------------
# Vérification de la cohérence temporelle
# --------------------------------------------------

df_survival_clean = (
    df_ckd
    .filter(
        (pl.col("creat_date") >= pl.col("birth_date"))
        & (pl.col("end_date") >= pl.col("creat_date"))
        & (pl.col("time_to_event") >= 0)
        & ~(
            (pl.col("event_irt") == 0)
            & (pl.col("time_to_event") == 0)
        )
    )
    .sort([
        "patient_id",
        "creat_date"
    ])
)


# ==========================================================
# RÉSUMÉ DE LA SÉLECTION DE LA COHORTE
# ==========================================================

summary = [
    ("Cohorte initiale", df_survival_long),
    ("Patients ≥ 16 ans", df_adult),
    ("Données biologiques disponibles", df_biology),
    ("DFG ≥ 30 mL/min/1.73 m²", df_ckd),
    ("Cohorte finale", df_survival_clean),
]

print("\n================ Sélection de la cohorte ================\n")

for label, df in summary:

    n_patients = (
        df
        .select(pl.col("patient_id").n_unique())
        .item()
    )

    print(
        f"{label:<35}"
        f"{n_patients:>5} patients | "
        f"{df.height:>6} observations"
    )

In [ ]:
# ==========================================================
# DESCRIPTION DE LA COHORTE LONGITUDINALE
# ==========================================================

# -----------------------------
# Statistiques au niveau patient
# -----------------------------

patient_summary = (
    df_survival_clean
    .group_by("patient_id")
    .agg(
        pl.len().alias("n_measures"),
        pl.col("time_to_event").max().alias("follow_up"),
        pl.col("event_irt").max().alias("event_irt")
    )
)

# -----------------------------
# Effectifs
# -----------------------------

n_rows = df_survival_clean.height
n_patients = patient_summary.height

# -----------------------------
# Mesures répétées
# -----------------------------

measures_stats = (
    patient_summary
    .select(
        pl.col("n_measures").median().alias("median"),
        pl.col("n_measures").min().alias("min"),
        pl.col("n_measures").max().alias("max")
    )
)

median_measures = measures_stats["median"][0]
min_measures = measures_stats["min"][0]
max_measures = measures_stats["max"][0]

# -----------------------------
# Suivi et événements
# -----------------------------

median_follow_up = (
    patient_summary
    .select(
        pl.col("follow_up").median()
    )
    .item()
)

irt_rate = (
    patient_summary
    .select(
        pl.col("event_irt").mean()
    )
    .item()
)

# ==========================================================
# RÉSUMÉ DE LA COHORTE
# ==========================================================

print("\n================ Description de la cohorte ================\n")

print(f"Nombre total d'observations : {n_rows}")
print(f"Nombre de patients          : {n_patients}")

print("\nMesures par patient")
print(f"  Médiane : {median_measures}")
print(f"  Minimum : {min_measures}")
print(f"  Maximum : {max_measures}")

print("\nSuivi")
print(f"  Temps médian de suivi : {median_follow_up:.2f} ans")

print("\nÉvénement IRT")
print(f"  Patients avec IRT : {irt_rate:.2%}")

In [ ]:
# ==========================================================
# CONSTRUCTION DES VARIABLES CLINIQUES DÉPENDANTES DU TEMPS
# ==========================================================

def build_dynamic_clinical(
    age_event_col: str,
    has_event_col: str,
    status_name: str,
    age_name: str
):
    """
    Construit deux variables longitudinales décrivant un événement clinique.

    Paramètres
    ----------
    age_event_col : str
        Nom de la colonne contenant l'âge de survenue de l'événement.

    has_event_col : str
        Indicateur indiquant si le patient a présenté l'événement.

    status_name : str
        Nom de la variable de statut créée.

    age_name : str
        Nom de la variable contenant l'âge de survenue.

    Variables créées
    ----------------
    status :
        0 = événement non survenu à la date de la visite
        1 = événement déjà survenu (âge connu)
        2 = événement présent mais âge de survenue inconnu

    age :
        âge auquel l'événement est survenu, renseigné
        uniquement lorsque l'événement est déjà connu
        à la date de la visite, sinon null.
    """

    status = (

        pl.when(
            pl.col(age_event_col).is_not_null()
            &
            (pl.col("age_creat") >= pl.col(age_event_col))
        )
        .then(pl.lit(1))

        .when(
            (pl.col(has_event_col) == 1)
            &
            pl.col(age_event_col).is_null()
        )
        .then(pl.lit(2))

        .otherwise(pl.lit(0))
        .alias(status_name)

    )

    age = (

        pl.when(
            pl.col(age_event_col).is_not_null()
            &
            (pl.col("age_creat") >= pl.col(age_event_col))
        )
        .then(pl.col(age_event_col))

        .otherwise(None)
        .alias(age_name)

    )

    return [status, age]

In [ ]:
# ==========================================================
# BASE DE DÉPART POUR LA CONSTRUCTION DU DATASET DYNAMIQUE
# ==========================================================

# La cohorte nettoyée est utilisée comme point de départ pour
# construire les différents snapshots longitudinaux (1, 2, …, k visites).

df_survival_dynamic = df_survival_clean

print(
    f"Base dynamique : "
    f"{df_survival_dynamic.select(pl.col('patient_id').n_unique()).item()} patients | "
    f"{df_survival_dynamic.height} observations"
)

In [ ]:
# ==========================================================
# CONSTRUCTION DES VARIABLES CLINIQUES DÉPENDANTES DU TEMPS
# ==========================================================

# Pour chaque visite, les variables cliniques sont mises à jour
# afin de refléter l'état du patient au moment de cette mesure.

df_survival_dynamic = (

    df_survival_dynamic

    .with_columns(

        # Hypertension

        build_dynamic_clinical(
            age_event_col="age_hypertension",
            has_event_col="has_hypertension",
            status_name="hypertension_at_t",
            age_name="age_hypertension_at_t"
        )

        +

        # Complication urologique

        build_dynamic_clinical(
            age_event_col="age_urologic_event",
            has_event_col="has_urologic_event",
            status_name="urologic_event_at_t",
            age_name="age_urologic_event_at_t"
        )

    )

)


In [ ]:
# ==========================================================
# CONTRÔLE DES VARIABLES DYNAMIQUES
# ==========================================================

# Vérification de la construction des variables dépendantes
# du temps sur les premières observations.

df_survival_dynamic.select([

    "patient_id",
    "creat_date",
    "age_creat",
    "age_hypertension",

    "has_hypertension",
    "hypertension_at_t",
    "age_hypertension_at_t",

    "has_urologic_event",
    "urologic_event_at_t",
    "age_urologic_event_at_t",

    "end_date",

    "time_to_event",
    "event_irt"

]).head(10)

### Séparation de la cohorte de développement et de validation externe

In [ ]:
# ==========================================================
# VALIDATION EXTERNE PAR CENTRE
# ==========================================================

# Centres réservés à la validation externe.
# Ils sont exclus de la phase de développement afin d'évaluer
# la capacité de généralisation du modèle sur des centres indépendants.

external_centers = ["09", "04"]

# Cohorte de développement
df_development = (
    df_survival_dynamic
    .filter(
        ~pl.col("center_id").is_in(external_centers)
    )
)

# Cohorte de validation externe
df_external_test = (
    df_survival_dynamic
    .filter(
        pl.col("center_id").is_in(external_centers)
    )
)

In [ ]:
# ==========================================================
# RÉSUMÉ DES COHORTES
# ==========================================================

def summarize_cohort(df, cohort_name):
    """Affiche les caractéristiques principales d'une cohorte."""

    n_patients = (
        df
        .select(
            pl.col("patient_id").n_unique()
        )
        .item()
    )

    patient_summary = (
        df
        .group_by("patient_id")
        .agg(
            pl.col("event_irt").max().alias("event_irt")
        )
    )

    n_events = (
        patient_summary
        .select(
            pl.col("event_irt").sum()
        )
        .item()
    )

    event_rate = (
        patient_summary
        .select(
            pl.col("event_irt").mean()
        )
        .item()
    )

    print(
        f"{cohort_name:<22}"
        f"{n_patients:>5} patients | "
        f"{df.height:>6} observations | "
        f"{n_events:>4} événements ({event_rate:.1%})"
    )


print("\n================ Validation externe ================\n")

summarize_cohort(
    df_development,
    "Développement"
)

summarize_cohort(
    df_external_test,
    "Validation externe"
)

In [ ]:
# ==========================================================
# SAUVEGARDE DES COHORTES
# ==========================================================

df_development.write_parquet(
    "My_data/adpkd_longitudinal_development.parquet"
)

df_external_test.write_parquet(
    "My_data/adpkd_longitudinal_external_test.parquet"
)

print("Cohortes sauvegardées avec succès.")